# UFO Sightings Analysis

Python pipeline for the NUFORC sighting reports dataset (1906–2014, 80,332 reports). Cleaning, geographic analysis, and LDA topic modeling. The cleaned data and pre-aggregated tables feed a Power BI dashboard.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10
RNG = 42

## Cleaning

The raw CSV is mostly clean but has a few issues that need handling:
- `longitude ` has a trailing space in the header
- `country` missing in 12% of rows; can be inferred from lat/lon
- `shape` missing in 2%; 29 source shapes need bucketing
- `latitude` is string, with one corrupted value
- `comments` contains HTML entities (`&#44;`, `&amp;`)
- Duration ranges from 1 ms to 97.8M seconds — most extremes are user errors

In [ ]:
df_raw = pd.read_csv("data_raw.csv", low_memory=False)
df_raw.columns = df_raw.columns.str.strip()
print(f"Shape: {df_raw.shape}")
print(df_raw.isna().sum().sort_values(ascending=False).head(8))

In [ ]:
# Shape bucketing: 29 source shapes → 10 families
SHAPE_FAMILIES = {
    "light": "Light/Orb", "flash": "Light/Orb", "flare": "Light/Orb",
    "disk": "Disk", "oval": "Disk", "dome": "Disk",
    "triangle": "Triangle", "chevron": "Triangle", "delta": "Triangle", "pyramid": "Triangle",
    "sphere": "Sphere/Circle", "circle": "Sphere/Circle", "round": "Sphere/Circle",
    "egg": "Sphere/Circle", "crescent": "Sphere/Circle",
    "fireball": "Fireball", "teardrop": "Fireball",
    "cigar": "Cigar/Cylinder", "cylinder": "Cigar/Cylinder",
    "rectangle": "Rectangle", "diamond": "Rectangle", "cross": "Rectangle",
    "cone": "Rectangle", "hexagon": "Rectangle",
    "formation": "Formation",
    "changing": "Changing", "changed": "Changing",
    "other": "Other/Unknown", "unknown": "Other/Unknown",
}

# Country bounding boxes
COUNTRY_BOXES = [
    ("us", 24.5, 49.4, -125.0, -66.9),    # CONUS
    ("us", 51.0, 71.5, -180.0, -129.0),   # Alaska
    ("us", 18.9, 22.3, -160.5, -154.7),   # Hawaii
    ("ca", 41.7, 83.1, -141.0, -52.6),
    ("gb", 49.9, 60.9, -8.6, 1.8),
    ("au", -43.6, -10.7, 113.0, 153.6),
    ("de", 47.3, 55.1, 5.9, 15.0),
]

def decode_html_text(s):
    if pd.isna(s): return ""
    s = html.unescape(str(s))
    s = re.sub(r"&#(\d+);", lambda m: chr(int(m.group(1))), s)
    return re.sub(r"\s+", " ", s).strip()

def infer_country(row):
    if isinstance(row["country"], str) and row["country"]: return row["country"]
    lat, lon = row["latitude"], row["longitude"]
    if pd.isna(lat) or pd.isna(lon): return "unknown"
    for name, lat_min, lat_max, lon_min, lon_max in COUNTRY_BOXES:
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return name
    return "other"

df = df_raw.copy()
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df = df.dropna(subset=["latitude"]).copy()

df["sighting_dt"] = pd.to_datetime(df["datetime"], errors="coerce")
df["posted_dt"] = pd.to_datetime(df["date posted"], errors="coerce")
df["year"] = df["sighting_dt"].dt.year
df["month"] = df["sighting_dt"].dt.month
df["day_of_week"] = df["sighting_dt"].dt.day_name()
df["hour"] = df["sighting_dt"].dt.hour
df["decade"] = (df["year"] // 10 * 10).astype("Int64")
df["report_lag_days"] = (df["posted_dt"] - df["sighting_dt"]).dt.days.clip(lower=0)

df["duration_sec"] = pd.to_numeric(df["duration (seconds)"], errors="coerce")
df["duration_reliable"] = df["duration_sec"].between(5, 7200).fillna(False)

df["shape"] = df["shape"].fillna("unknown").str.lower().str.strip()
df["shape_family"] = df["shape"].map(SHAPE_FAMILIES).fillna("Other/Unknown")

df["country"] = df["country"].astype(str).str.lower().replace("nan", np.nan)
df["country"] = df.apply(infer_country, axis=1)

df["state"] = df["state"].astype(str).str.lower().str.strip().replace("nan", np.nan)
df["city"] = df["city"].astype(str).str.lower().str.strip()
df["comments_clean"] = df["comments"].apply(decode_html_text)
df["era"] = np.where(df["year"] < 1998, "Pre-NUFORC web (retrospective)", "NUFORC web era (1998+)")

print(f"Final rows: {len(df):,}")
print(f"Reliable durations: {df['duration_reliable'].sum():,} ({df['duration_reliable'].mean():.1%})")

## Reports over time

NUFORC's web reporting form launched in 1998. Pre-1998 reports came in by mail or phone. The data structure changes at that point.

In [ ]:
yearly = df.groupby("year").size().reset_index(name="reports")
yearly = yearly[yearly["year"] >= 1940].astype({"year": int})

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(yearly["year"], yearly["reports"], color="#2c3e50", width=0.85)
ax.axvline(1997.5, color="#c0392b", linestyle="--", linewidth=2,
           label="NUFORC launches web reporting (1998)")
ax.set_xlabel("Year"); ax.set_ylabel("Reports")
ax.set_title("UFO reports by year")
ax.legend()
plt.tight_layout(); plt.show()

print(f"Pre-1998 total: {yearly[yearly['year'] < 1998]['reports'].sum():,} across {(1998-1940)} years")
print(f"1998+ total:    {yearly[yearly['year'] >= 1998]['reports'].sum():,} across {(2015-1998)} years")

Pre-1998 averaged 174 reports/year. Post-1998 averaged 4,000/year. The platform launch dwarfs every other variable in the dataset.

## Hour of day

In [ ]:
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
heat = df.dropna(subset=["hour","day_of_week"]).groupby(["day_of_week","hour"]).size().reset_index(name="n")
heat["hour"] = heat["hour"].astype(int)
pivot = heat.pivot(index="day_of_week", columns="hour", values="n").reindex(dow_order)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap="YlOrRd", cbar_kws={"label": "Reports"}, ax=ax,
            linewidths=0.3, linecolor="white")
ax.set_xlabel("Hour of day"); ax.set_ylabel("")
ax.set_title("Reports by hour and day of week")
plt.tight_layout(); plt.show()

print(f"Peak hour: 21:00 ({df[df['hour']==21].shape[0]:,} reports)")
print(f"Peak cell: Saturday 22:00 ({pivot.loc['Saturday', 22]:,} reports)")

Sightings cluster between 9 PM and midnight, with Friday/Saturday peaks. People file reports when they are outside in the dark.

## Geography (US)

Raw counts mostly track population. Per-capita rates against 2010 Census numbers reveal more interesting patterns.

In [ ]:
state_pop = pd.read_csv("state_populations.csv")
state_pop["state_code"] = state_pop["state_code"].str.lower()

us = df[df["country"] == "us"].copy()
state_summary = us.groupby("state").size().reset_index(name="reports")
state_summary = state_summary.merge(state_pop, left_on="state", right_on="state_code", how="left")
state_summary = state_summary.dropna(subset=["population_2010"])
state_summary["reports_per_100k"] = state_summary["reports"] / state_summary["population_2010"] * 100000
state_summary = state_summary.sort_values("reports_per_100k", ascending=False)

fig, ax = plt.subplots(figsize=(7, 11))
colors = ["#c0392b" if x >= 40 else "#34495e" if x >= 25 else "#7f8c8d"
          for x in state_summary["reports_per_100k"][::-1]]
ax.barh(state_summary["state_name"][::-1], state_summary["reports_per_100k"][::-1], color=colors)
ax.set_xlabel("Reports per 100,000 residents (2010 Census)")
ax.set_title("UFO reports per capita by US state")
plt.tight_layout(); plt.show()

Washington tops at 63 reports per 100k. Louisiana at the bottom with 13. The top of the list is Pacific Northwest, Mountain West, and Northern New England. The bottom is the Deep South.

The cause isn't derivable from the data alone — possible factors include lower light pollution out West, military aviation corridors, and NUFORC's Seattle headquarters creating local awareness.

## Shape evolution

In [ ]:
shape_dec = (
    df.dropna(subset=["decade"]).groupby(["decade","shape_family"]).size()
    .reset_index(name="reports")
)
shape_dec["decade_total"] = shape_dec.groupby("decade")["reports"].transform("sum")
shape_dec["share"] = shape_dec["reports"] / shape_dec["decade_total"]
shape_dec["decade"] = shape_dec["decade"].astype(int)
shape_dec = shape_dec[shape_dec["decade"] >= 1950]

pivot = shape_dec.pivot(index="decade", columns="shape_family", values="share").fillna(0)
col_order = ["Light/Orb","Sphere/Circle","Triangle","Disk","Fireball",
             "Cigar/Cylinder","Formation","Rectangle","Changing","Other/Unknown"]
pivot = pivot[[c for c in col_order if c in pivot.columns]]

palette = ["#f1c40f","#3498db","#e74c3c","#9b59b6","#e67e22",
           "#1abc9c","#34495e","#95a5a6","#7f8c8d","#bdc3c7"]
fig, ax = plt.subplots(figsize=(11, 5))
pivot.plot.area(ax=ax, color=palette[:len(pivot.columns)], alpha=0.85, stacked=True)
ax.set_xlabel("Decade"); ax.set_ylabel("Share of reports")
ax.set_title("Shape families over time")
ax.set_ylim(0, 1)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout(); plt.show()

Disk peaked at ~40% in the 1950s–70s and is now ~11%. Light/Orb has grown to 24%. Triangle was near zero in the 1950s and is now ~12%.

Either the actual shapes of UFOs have changed over 60 years, or what people report has shifted with cultural framing. The second is more likely.

## Text mining

Tokenize the cleaned comments, then look at distinctive vocabulary by shape family using TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

CUSTOM_STOPWORDS = set('''
a about above after again against all am an and any are aren as at be
because been before being below between both but by could did do does
doing don down during each few for from further had has have having he
her here hers herself him himself his how i if in into is it its itself
just me more most my myself no nor not now of off on once only or other
our ours ourselves out over own same she should so some such than that
the their theirs them themselves then there these they this those
through to too under until up very was we were what when where which
while who whom why will with you your yours yourself yourselves
would also like get got see seen saw look looked looking would could
one two three four five six seven eight nine ten time times back
went came going around went thing things something anything
ufo ufos object objects sighting witness report nuforc pd note us
http www com html
'''.split())

def tokenize(text):
    if not isinstance(text, str): return []
    return [t for t in re.findall(r"[a-z]+", text.lower())
            if len(t) >= 3 and t not in CUSTOM_STOPWORDS]

docs = df[df["comments_clean"].fillna("").str.len() >= 20].copy()
print(f"Reports with usable comments: {len(docs):,}")

In [ ]:
cv = CountVectorizer(tokenizer=tokenize, lowercase=False, min_df=20, token_pattern=None)
X = cv.fit_transform(docs["comments_clean"].fillna(""))
freqs = pd.DataFrame({
    "word": cv.get_feature_names_out(),
    "count": np.asarray(X.sum(axis=0)).ravel(),
}).sort_values("count", ascending=False)

print("Top 15 words overall:")
print(freqs.head(15).to_string(index=False))

In [ ]:
# TF-IDF: most distinctive word per shape family
shape_docs = docs.groupby("shape_family")["comments_clean"].apply(lambda s: " ".join(s.fillna(""))).reset_index()
tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, min_df=2,
                        max_df=0.95, token_pattern=None, max_features=2000)
T = tfidf.fit_transform(shape_docs["comments_clean"])
vocab = tfidf.get_feature_names_out()

print("Most distinctive words by shape family:")
for i, family in enumerate(shape_docs["shape_family"]):
    row = T[i].toarray().ravel()
    top = [vocab[j] for j in row.argsort()[::-1][:5]]
    print(f"  {family:18s} -> {', '.join(top)}")

The Light/Orb row is notable: distinctive words include `iss` (International Space Station) and `hbccufo` (HBCC UFO Research, an organization that frequently tags misidentified satellites). A meaningful share of modern "light" reports are likely ISS passes.

## LDA topic modeling

Subsample to balance the corpus across years, then fit an 8-topic LDA model.

In [ ]:
N_TOPICS = 8
sample = (
    docs.groupby("year", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 500), random_state=RNG))
)
print(f"LDA training sample: {len(sample):,} reports")

cv_lda = CountVectorizer(tokenizer=tokenize, lowercase=False, min_df=20,
                          max_df=0.6, token_pattern=None, max_features=3000)
X_lda = cv_lda.fit_transform(sample["comments_clean"].fillna(""))

lda = LatentDirichletAllocation(n_components=N_TOPICS, learning_method="online",
                                  random_state=RNG, max_iter=20, n_jobs=-1)
lda.fit(X_lda)

vocab_lda = cv_lda.get_feature_names_out()
print("\nTopics — top 8 words each:")
for i, topic in enumerate(lda.components_):
    top = [vocab_lda[j] for j in topic.argsort()[::-1][:8]]
    print(f"  Topic {i}: {', '.join(top)}")

Topic interpretations:

| Topic | Theme |
|---|---|
| 0 | Close encounters (house, hovering, area, car) |
| 1 | Orange triangle formations |
| 2 | Multi-color flashing craft |
| 3 | Star-like fast objects (likely satellites) |
| 4 | Green fireballs |
| 5 | Black triangle / stealth-aircraft-adjacent |
| 6 | Generic bright lights, directional |
| 7 | Trajectory descriptions |

In [ ]:
# Topic prevalence over time
X_full = cv_lda.transform(docs["comments_clean"].fillna(""))
docs["dominant_topic"] = lda.transform(X_full).argmax(axis=1)

topic_dec = (
    docs.dropna(subset=["decade"]).groupby(["decade","dominant_topic"]).size()
    .reset_index(name="reports")
)
topic_dec["decade_total"] = topic_dec.groupby("decade")["reports"].transform("sum")
topic_dec["share"] = topic_dec["reports"] / topic_dec["decade_total"]
topic_dec["decade"] = topic_dec["decade"].astype(int)
topic_dec = topic_dec[topic_dec["decade"] >= 1970]

labels = {}
for i, topic in enumerate(lda.components_):
    top3 = [vocab_lda[j] for j in topic.argsort()[::-1][:3]]
    labels[i] = f"T{i}: {', '.join(top3)}"
topic_dec["topic_label"] = topic_dec["dominant_topic"].map(labels)

pivot = topic_dec.pivot(index="decade", columns="topic_label", values="share").fillna(0)
fig, ax = plt.subplots(figsize=(11, 5))
pivot.plot(ax=ax, marker="o", linewidth=1.5, alpha=0.85)
ax.set_xlabel("Decade"); ax.set_ylabel("Share of reports")
ax.set_title("LDA topic prevalence by decade")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

The orange-triangle topic (T1) rises sharply post-1990. Plausibly tied to consumer sky lanterns and recreational drones becoming widespread.

The close-encounter topic (T0) declines. Modern reports describe objects further away, more sky-watching, less intimate.

## Output

Running the standalone scripts (`prep.py`, `analysis.py`, `text_mining.py`) writes aggregated tables to `powerbi_data/` for the dashboard. See `powerbi_build_guide.md` for the build steps.